# Sistem Rekomendasi Smartphone Menggunakan Analisis Sentimen Berbasis Aspek
### Perbandingan Model IndoBERT dan BiLSTM (aspek: Baterai, Kamera, Performa, Harga)

Notebook ini memuat **seluruh alur penelitian secara berurutan**, dari pengumpulan data sampai tabel rekomendasi akhir:

| Tahap | Isi | Keluaran utama |
|---|---|---|
| 1 | Pengumpulan data (scraping komentar YouTube) | `dataset_mentah_multi_smartphone.csv` |
| 2 | Preprocessing teks | `dataset_preprocessed_smartphone.csv` |
| 3 | Pelabelan ABSA (4 aspek x 4 label) + eksplorasi label | tabel distribusi label |
| 4 | Pembagian data latih/uji 80:20 | `Data_Training_ABSA.csv`, `Data_Testing_ABSA.csv` |
| 5 | Model BiLSTM per aspek | metrik uji BiLSTM |
| 6 | Model IndoBERT per aspek | metrik uji IndoBERT |
| 7 | Perbandingan BiLSTM vs IndoBERT | tabel + grafik perbandingan |
| 7b | Confusion matrix dan uji McNemar | `confusion_matrix.png`, `uji_mcnemar.csv` |
| 8 | Prediksi sentimen seluruh komentar oleh model terpilih | `absa_prediksi_<model>.csv` |
| 9 | Ekstraksi merek/model smartphone dari komentar | `audit_atribusi.csv` |
| 10 | Sistem rekomendasi (Top Merek & Top Model per aspek) | `rekomendasi_smartphone.xlsx` |
| 10b-10d | Rekomendasi berbobot prioritas pengguna, demo interaktif, ekspor data antarmuka | `contoh_rekomendasi_berbobot.csv`, `data_ui.json` |
| 11 | Ekspor seluruh hasil (ZIP) | `hasil_penelitian.zip` |

**Cara pakai:** *Runtime > Change runtime type > T4 GPU*, lalu jalankan sel dari atas ke bawah.
File yang perlu diunggah (jika tidak melakukan scraping ulang): `dataset_mentah_multi_smartphone.csv` dan `absa_label_smartphone.csv`.

**Perubahan dibanding notebook lama (dan alasannya)**
1. API key YouTube tidak lagi ditulis di kode (diminta lewat `getpass`), dan token Hugging Face dihapus karena `indobert-base-p1` bersifat publik.
2. Preprocessing **tidak diubah**, hanya kolom teks asli (`Teks_Ulasan`) tidak dibuang lagi. Teks asli inilah yang dipakai untuk mengenali nama produk pada tahap 9.
3. BiLSTM diperbaiki. Sebelumnya teks diubah menjadi satu vektor hitungan kata lalu dimasukkan sebagai urutan sepanjang 1, sehingga LSTM tidak membaca urutan kata. Sekarang memakai Embedding dan urutan kata sebenarnya.
4. Ada **data validasi** (10% dari data latih) untuk memilih epoch terbaik. Data uji tidak lagi dipakai untuk memilih model.
5. Selain akurasi, dilaporkan **macro-F1** dan laporan per kelas, karena label "-" mencakup 78-90% data sehingga akurasi saja menyesatkan.
6. Tabel rekomendasi dihitung dari **hasil prediksi model**, bukan dari kolom label di CSV.
7. BiLSTM berjalan sebagai proses terpisah (CPU) agar tidak menghabiskan memori GPU untuk IndoBERT, dan hasil tiap aspek disimpan sebagai cache sehingga tahap yang sudah selesai tidak perlu diulang.

## 0. Persiapan

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn openpyxl

In [ ]:
import os, re, gc, json, random, collections, shutil, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings("ignore")

# ------------------------- PENGATURAN UTAMA -------------------------
SEED = 42
ASPEK = ["Baterai", "Kamera", "Performa", "Harga"]
KELAS = ["-", "Negatif", "Netral", "Positif"]        # '-' = aspek tidak dibahas
K2I = {k: i for i, k in enumerate(KELAS)}

JALANKAN_SCRAPING = False        # True = ambil ulang komentar dari YouTube (butuh API key)
MODEL_REKOMENDASI = "IndoBERT"   # model yang hasil prediksinya dipakai untuk rekomendasi: "IndoBERT" / "BiLSTM"
BILSTM_CLASS_WEIGHT = True       # bobot kelas pada BiLSTM agar tidak hanya menebak '-'
SIMPAN_MODEL_INDOBERT = False    # True = simpan bobot model ke disk Colab (+/- 440 MB per aspek)
BATCH_INDOBERT = 8               # kecilkan ke 4 jika GPU kehabisan memori
PAKAI_CACHE = True               # True = lewati aspek yang hasilnya sudah tersimpan (folder hasil/cache); False = latih ulang semua
PAKAI_GOOGLE_DRIVE = False       # True = simpan hasil ke Google Drive agar tidak hilang saat sesi Colab terputus/reset

FILE_MENTAH = "dataset_mentah_multi_smartphone.csv"
FILE_LABEL = "absa_label_smartphone.csv"
FOLDER_HASIL = "hasil"
if PAKAI_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_HASIL = "/content/drive/MyDrive/skripsi_absa/hasil"
os.makedirs(FOLDER_HASIL, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

DAFTAR_VIDEO = ["qeWDOOfbIbI", "vfp0r-lFm1k", "UOTT4xudY70", "0WTTYcWTJ4A", "pzzRk-dun7g",
                "KNdbPdZjlOU", "J7tekVDdUhg", "LRE_WGxHpGs", "ybK8FRc6nlw", "bwF7wNZjhLE"]

## 1. Pengumpulan Data (Scraping YouTube)
Komentar diambil dari 10 video review/komparasi smartphone memakai YouTube Data API v3 (komentar > 25 karakter, maksimal 400 per video).
Jika `JALANKAN_SCRAPING = False`, sel ini hanya membaca file hasil scraping yang sudah ada (unggah lewat sel di bawah).

In [ ]:
if JALANKAN_SCRAPING:
    from getpass import getpass
    from googleapiclient.discovery import build
    API_KEY = getpass("Masukkan YouTube API key (tidak disimpan di notebook): ")
    youtube = build("youtube", "v3", developerKey=API_KEY)

    def scrape_banyak_video(daftar_id):
        semua_komentar = []
    
        for vid_id in daftar_id:
            print(f"Sedang mengambil komentar dari video ID: {vid_id}...")
            komentar_per_video = 0
    
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=vid_id,
                maxResults=100,
                textFormat="plainText"
            )
    
            while request:
                try:
                    response = request.execute()
                    for item in response['items']:
                        comment = item['snippet']['topLevelComment']['snippet']['textDisplay']
                        author = item['snippet']['topLevelComment']['snippet']['authorDisplayName']
    
                        # Menyaring ulasan agar bukan sekadar spam/komentar pendek
                        if len(comment) > 25:
                            semua_komentar.append({
                                'Video_ID': vid_id,
                                'Username': author,
                                'Teks_Ulasan': comment
                            })
                            komentar_per_video += 1
    
                    # Batasan per video (misal maksimal 400 komentar per video)
                    if komentar_per_video >= 400:
                        break
    
                    request = youtube.commentThreads().list_next(request, response)
                except Exception as e:
                    print(f"Kendala pada video {vid_id}: {e}")
                    break
    
            print(f"Selesai untuk {vid_id}, terkumpul {komentar_per_video} ulasan.\n")
    
        return semua_komentar

    data_ulasan = scrape_banyak_video(DAFTAR_VIDEO)
    df_mentah = pd.DataFrame(data_ulasan)
    df_mentah.to_csv(FILE_MENTAH, index=False, encoding="utf-8")
else:
    if not os.path.exists(FILE_MENTAH):
        from google.colab import files
        print("Unggah file:", FILE_MENTAH)
        files.upload()
    df_mentah = pd.read_csv(FILE_MENTAH)

print(f"Total komentar mentah: {len(df_mentah)} dari {df_mentah['Video_ID'].nunique()} video")
df_mentah.head()

## 2. Preprocessing
Tahapan: hapus mention/URL, normalisasi kata berulang, case folding, normalisasi slang, lalu buang komentar yang tersisa < 3 kata.
Fungsi pembersihan **sama persis dengan notebook lama**, sehingga kolom `Teks_Bersih` identik dengan yang dipakai saat pelabelan.

Catatan penting: huruf non-abjad (termasuk angka) dihapus dari `Teks_Bersih`. Itu tidak masalah untuk klasifikasi sentimen, tetapi membuat nama seri HP hilang
(mis. "iPhone 15 Pro" menjadi "iphone pro"). Karena itu kolom `Teks_Ulasan` (teks asli, masih ada angka) **dipertahankan** untuk tahap ekstraksi produk.

In [ ]:
# 2. Kamus normalisasi slang tingkat lanjut (Advanced)
kamus_slang = {
    # Singkatan Umum
    'yg': 'yang', 'nya': '', 'aja': 'saja', 'aj': 'saja', 'jga': 'juga', 'jg': 'juga',
    'udh': 'sudah', 'udah': 'sudah', 'dah': 'sudah', 'blm': 'belum', 'blom': 'belum',
    'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'ndak': 'tidak', 'gk': 'tidak', 'g': 'tidak',
    'klo': 'kalau', 'kalo': 'kalau', 'kl': 'kalau',
    'dgn': 'dengan', 'dg': 'dengan', 'krn': 'karena', 'karna': 'karena', 'krna': 'karena',
    'tp': 'tapi', 'tpi': 'tapi', 'dr': 'dari', 'dri': 'dari',
    'dn': 'dan', 'jdi': 'jadi', 'jd': 'jadi', 'utk': 'untuk', 'buat': 'untuk',
    'pake': 'pakai', 'pke': 'pakai', 'make': 'pakai', 'dpt': 'dapat',
    'lbh': 'lebih', 'bgt': 'banget', 'bgtu': 'begitu', 'bg': 'bang', 'bng': 'bang',
    'emng': 'memang', 'emg': 'memang', 'bener': 'benar', 'beneran': 'benar',
    'dlu': 'dulu', 'org': 'orang', 'sma': 'sama', 'sm': 'sama', 'ko': 'kok', 'sih': '', 'nih': '', 'deh': '', 'tuh': '',

    # Istilah Gadget & Sentimen
    'hp': 'handphone', 'hape': 'handphone', 'ponsel': 'handphone',
    'ip': 'iphone', 'ipon': 'iphone', 'aipon': 'iphone', 'pm': 'pro max',
    'samsoeng': 'samsung', 'somay': 'xiaomi', 'mending': 'lebih baik',
    'batre': 'baterai', 'batere': 'baterai', 'battery': 'baterai', 'bh': 'battery health', 'sot': 'screen on time',
    'bocorr': 'bocor', 'boross': 'boros', 'awet': 'tahan lama', 'irit': 'tahan lama',
    'cas': 'isi daya', 'ngecas': 'isi daya', 'charging': 'isi daya', 'casan': 'pengisi daya', 'charger': 'pengisi daya',
    'kam': 'kamera', 'cam': 'kamera', 'jepretan': 'kamera', 'vidio': 'video',
    'burik': 'jelek', 'bapuk': 'jelek', 'busuk': 'jelek', 'bening': 'bagus', 'tajem': 'tajam',
    'overheat': 'panas berlebih', 'anget': 'hangat', 'panass': 'panas',
    'gaming': 'main game', 'ngegame': 'main game', 'lag': 'patah patah', 'frame drop': 'patah patah', 'smooth': 'lancar',
    'worth': 'sepadan', 'worthed': 'sepadan', 'worthit': 'sepadan', 'cuan': 'untung',
    'jt': 'juta', 'jtan': 'jutaan', 'mahal': 'mahal', 'kemahalan': 'sangat mahal', 'murmer': 'murah meriah'
}

def clean_text_advanced(text):
    if not isinstance(text, str):
        return ""

    # Hapus mention (@username) dan URL
    text = re.sub(r'@[A-Za-z0-9_-]+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Penanganan Kata Ulang Angka (misal: "masing2" -> "masing masing")
    text = re.sub(r'([a-zA-Z]+)2', r'\1 \1', text)

    # Penanganan huruf berulang (misal: "puanasss" -> "puannas")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Beri spasi pada tanda baca agar tidak lengket (misal: "bagus,tapi" -> "bagus , tapi")
    text = re.sub(r'([.,!?()])', r' \1 ', text)

    # Hapus semua karakter selain huruf abjad (tanda baca, emoji, angka dihilangkan)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Case folding (huruf kecil)
    text = text.lower()

    # Normalisasi kata slang (Kata per Kata)
    words = text.split()
    normalized_words = [kamus_slang.get(w, w) for w in words]
    text = ' '.join(normalized_words)

    # Hapus spasi ganda yang tersisa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_mentah["Teks_Bersih"] = df_mentah["Teks_Ulasan"].apply(clean_text_advanced)
df_clean = df_mentah[df_mentah["Teks_Bersih"].str.split().str.len() >= 3].copy().reset_index(drop=True)
# Kolom Teks_Ulasan SENGAJA tidak dihapus (dipakai untuk mengenali nama produk)

df_clean.to_csv(f"{FOLDER_HASIL}/dataset_preprocessed_smartphone.csv", index=False, encoding="utf-8")
print(f"Komentar mentah: {len(df_mentah)} -> setelah filter >= 3 kata: {len(df_clean)}")
pd.set_option("display.max_colwidth", 90)
df_clean[["Teks_Ulasan", "Teks_Bersih"]].head(5)

## 3. Pelabelan ABSA dan Eksplorasi Label
Setiap komentar diberi label untuk 4 aspek: **Positif**, **Negatif**, **Netral**, atau **-** (aspek tidak dibahas).
Label dibuat dengan bantuan LLM (Claude) atas `Teks_Bersih`; sebutkan metode ini apa adanya di Bab 3 (metodologi) dan keterbatasan.

Sel berikut memuat file label, **memastikan baris-barisnya cocok satu per satu** dengan hasil preprocessing di atas (Video_ID, Username, dan Teks_Bersih), lalu menempelkan label ke tabel utama.

In [ ]:
if not os.path.exists(FILE_LABEL):
    from google.colab import files
    print("Unggah file:", FILE_LABEL)
    files.upload()
df_label = pd.read_csv(FILE_LABEL)

# Validasi keselarasan: jika salah satu gagal, urutan/isi data tidak sama dengan saat pelabelan
assert len(df_label) == len(df_clean), f"Jumlah baris beda: label={len(df_label)} vs preprocessing={len(df_clean)}"
assert (df_label["Video_ID"].values == df_clean["Video_ID"].values).all(), "Video_ID tidak sejajar"
assert (df_label["Username"].values == df_clean["Username"].values).all(), "Username tidak sejajar"
assert (df_label["Teks_Bersih"].values == df_clean["Teks_Bersih"].values).all(), "Teks_Bersih tidak sama"
for a in ASPEK:
    assert set(df_label[a].unique()) <= set(KELAS), f"Label tak dikenal pada {a}: {df_label[a].unique()}"

df = df_clean.copy()
for a in ASPEK:
    df[a] = df_label[a].values
print("Keselarasan label OK:", len(df), "baris")

# Distribusi label per aspek
dist = pd.DataFrame({a: df[a].value_counts() for a in ASPEK}).reindex(KELAS).fillna(0).astype(int)
dist.loc["Total"] = dist.sum()
print("\nDistribusi label per aspek:"); print(dist)
base = {a: round(100 * (df[a] == "-").mean(), 1) for a in ASPEK}
print("\nAkurasi jika model SELALU menebak '-' (baseline mayoritas), %:", base)
print("Komentar yang membahas minimal satu aspek:", int((df[ASPEK] != "-").any(axis=1).sum()), "dari", len(df))

ax = dist.drop("Total").T.plot(kind="bar", stacked=True, figsize=(7, 3.5), rot=0)
ax.set_ylabel("Jumlah komentar"); ax.set_title("Distribusi label per aspek"); plt.tight_layout()
plt.savefig(f"{FOLDER_HASIL}/distribusi_label.png", dpi=150); plt.show()

## 4. Pembagian Data (80% Latih, 20% Uji)
Pembagian sama dengan notebook lama (`random_state=42`), sehingga data uji identik. Dari data latih, 10% akan dipisah lagi sebagai **data validasi** di dalam tahap pelatihan
(untuk memilih epoch terbaik), sehingga data uji benar-benar murni untuk evaluasi akhir.

In [ ]:
data_latih, data_uji = train_test_split(df, test_size=0.20, random_state=SEED)
df["Split"] = "train"
df.loc[data_uji.index, "Split"] = "test"

print(f"Total Keseluruhan : {len(df)} baris")
print(f"Data Training (80%): {len(data_latih)} baris")
print(f"Data Testing (20%) : {len(data_uji)} baris")

cols_simpan = ["Video_ID", "Username", "Teks_Bersih"] + ASPEK
data_latih[cols_simpan].to_csv(f"{FOLDER_HASIL}/Data_Training_ABSA.csv", index=False)
data_uji[cols_simpan].to_csv(f"{FOLDER_HASIL}/Data_Testing_ABSA.csv", index=False)

# Tempat menyimpan hasil evaluasi dan prediksi dari kedua model
hasil_eval = []        # satu baris per (model, aspek)
pred_all = {}          # pred_all[model] = DataFrame prediksi seluruh komentar (kolom = aspek)

## 5. Model BiLSTM (per aspek)
Arsitektur: `Embedding(128)` -> `BiLSTM(64)` -> `Dropout` -> `BiLSTM(32)` -> `Dropout` -> `Dense(64)` -> `Softmax(4 kelas)`.
Teks diubah menjadi urutan indeks kata (panjang maksimal 60 kata; kosakata dibangun **hanya dari data latih**). Pelatihan berhenti otomatis (*early stopping*) berdasarkan loss validasi.

**Mengapa BiLSTM dijalankan sebagai proses terpisah?** TensorFlow cenderung menguasai hampir seluruh memori GPU dan tidak melepaskannya sampai proses selesai.
Jika BiLSTM berjalan di dalam notebook, IndoBERT bisa gagal karena memori GPU penuh. Dengan proses terpisah (CPU saja), memori GPU tetap bebas untuk IndoBERT,
dan Anda tidak perlu me-restart runtime atau mengganti jenis runtime. Hasil tiap aspek disimpan di `hasil/cache` sehingga tidak hilang jika sesi terputus.

In [ ]:
%%writefile bilstm_worker.py
# Dijalankan sebagai proses terpisah (CPU saja) agar TensorFlow tidak menguasai memori GPU
# yang dibutuhkan IndoBERT. Hasil disimpan per aspek di <folder>/cache.
import os, sys, json, collections
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

FOLDER, PAKAI_CACHE, CLASS_WEIGHT = sys.argv[1], sys.argv[2] == "1", sys.argv[3] == "1"
SEED = 42
ASPEK = ["Baterai", "Kamera", "Performa", "Harga"]
KELAS = ["-", "Negatif", "Netral", "Positif"]
K2I = {k: i for i, k in enumerate(KELAS)}
MAXLEN, MAXVOCAB = 60, 5000
TAG = f"cw{int(CLASS_WEIGHT)}"

df = pd.read_csv(f"{FOLDER}/bilstm_input.csv")
latih, uji = df[df["Split"] == "train"], df[df["Split"] == "test"]
os.makedirs(f"{FOLDER}/cache", exist_ok=True)

def bangun_kosakata(teks_list):
    c = collections.Counter(w for t in teks_list for w in str(t).split())
    return {w: i + 2 for i, (w, _) in enumerate(c.most_common(MAXVOCAB - 2))}    # 0=padding, 1=OOV

def ke_urutan(teks_list, vocab):
    X = np.zeros((len(teks_list), MAXLEN), dtype="int32")
    for i, t in enumerate(teks_list):
        ids = [vocab.get(w, 1) for w in str(t).split()][:MAXLEN]
        X[i, :len(ids)] = ids
    return X

def bangun_bilstm(n_vocab, n_kelas):
    inp = tf.keras.Input(shape=(MAXLEN,))
    x = tf.keras.layers.Embedding(n_vocab, 128, mask_zero=True)(inp)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32))(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    out = tf.keras.layers.Dense(n_kelas, activation="softmax")(x)
    m = tf.keras.Model(inp, out)
    m.compile(loss="sparse_categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(1e-3), metrics=["accuracy"])
    return m

for aspek in ASPEK:
    f_ev = f"{FOLDER}/cache/bilstm_{TAG}_{aspek}_eval.json"
    f_pr = f"{FOLDER}/cache/bilstm_{TAG}_{aspek}_pred.csv"
    if PAKAI_CACHE and os.path.exists(f_ev) and os.path.exists(f_pr):
        print(f"[cache] BiLSTM {aspek} dimuat dari cache, dilewati", flush=True)
        continue

    print(f"\n=== MENJALANKAN MODEL BILSTM UNTUK ASPEK: {aspek.upper()} ===", flush=True)
    tf.keras.utils.set_random_seed(SEED)
    tr, va = train_test_split(latih, test_size=0.10, random_state=SEED, stratify=latih[aspek])
    vocab = bangun_kosakata(tr["Teks_Bersih"])
    X_tr, X_va, X_te, X_all = (ke_urutan(d["Teks_Bersih"], vocab) for d in (tr, va, uji, df))
    y_tr, y_va, y_te = (d[aspek].map(K2I).values for d in (tr, va, uji))

    cw = None
    if CLASS_WEIGHT:
        cnt = np.bincount(y_tr, minlength=len(KELAS))
        cw = {i: float(min(len(y_tr) / (len(KELAS) * max(c, 1)), 10.0)) for i, c in enumerate(cnt)}

    model = bangun_bilstm(len(vocab) + 2, len(KELAS))
    model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=25, batch_size=32, verbose=0, class_weight=cw,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])

    y_pred = model.predict(X_te, verbose=0).argmax(1)
    acc = float(accuracy_score(y_te, y_pred))
    f1m = float(f1_score(y_te, y_pred, average="macro", labels=range(len(KELAS)), zero_division=0))
    print(classification_report(y_te, y_pred, labels=range(len(KELAS)), target_names=KELAS, digits=3, zero_division=0))
    rep = classification_report(y_te, y_pred, labels=range(len(KELAS)), target_names=KELAS, zero_division=0, output_dict=True)
    print(f"[HASIL] BiLSTM {aspek}: akurasi={acc*100:.2f}% | macro-F1={f1m:.3f}", flush=True)

    rec = {"Model": "BiLSTM", "Aspek": aspek, "Akurasi": acc, "Macro-F1": f1m,
           **{f"F1 {k}": float(rep[k]["f1-score"]) for k in KELAS}}
    pred = [KELAS[i] for i in model.predict(X_all, verbose=0).argmax(1)]
    json.dump(rec, open(f_ev, "w"))
    pd.DataFrame({"Idx": df["Idx"].values, aspek: pred}).to_csv(f_pr, index=False)
    del model; tf.keras.backend.clear_session()

print("\nBiLSTM selesai.", flush=True)

In [ ]:
import subprocess, sys

# Data masukan untuk BiLSTM: urutan baris latih/uji sama dengan pembagian di tahap 4
inp = df.loc[list(data_latih.index) + list(data_uji.index), ["Teks_Bersih", "Split"] + ASPEK].copy()
inp.insert(0, "Idx", inp.index)
inp.to_csv(f"{FOLDER_HASIL}/bilstm_input.csv", index=False)

proc = subprocess.Popen(
    [sys.executable, "-u", "bilstm_worker.py", FOLDER_HASIL, "1" if PAKAI_CACHE else "0", "1" if BILSTM_CLASS_WEIGHT else "0"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for baris in proc.stdout:
    print(baris, end="")
assert proc.wait() == 0, "Proses BiLSTM gagal, lihat pesan di atas."

# Muat hasil BiLSTM ke notebook
TAG_BL = f"cw{int(BILSTM_CLASS_WEIGHT)}"
pred_all["BiLSTM"] = pd.DataFrame(index=df.index)
for aspek in ASPEK:
    hasil_eval.append(json.load(open(f"{FOLDER_HASIL}/cache/bilstm_{TAG_BL}_{aspek}_eval.json")))
    p = pd.read_csv(f"{FOLDER_HASIL}/cache/bilstm_{TAG_BL}_{aspek}_pred.csv").set_index("Idx")[aspek]
    pred_all["BiLSTM"][aspek] = p.reindex(df.index).values
print("\nHasil BiLSTM dimuat:", [r["Aspek"] for r in hasil_eval if r["Model"] == "BiLSTM"])

## 6. Model IndoBERT (per aspek)
Model: `indobenchmark/indobert-base-p1` (fine-tuning penuh, 4 epoch, learning rate 2e-5, batch 8, mixed precision).
Epoch terbaik dipilih berdasarkan **macro-F1 pada data validasi**, bukan data uji. Memori GPU dibersihkan setiap selesai satu aspek (`gc.collect()`). Hasil tiap aspek disimpan ke `hasil/cache`; jika sesi terputus, jalankan ulang sel ini dan aspek yang sudah selesai akan dilewati (`PAKAI_CACHE = True`).

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)

def ke_dataset(frame):
    d = Dataset.from_pandas(frame[["Teks_Bersih", "y"]].rename(columns={"Teks_Bersih": "text", "y": "label"}),
                            preserve_index=False)
    return d.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)

def metrik(p):
    pred = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, pred),
            "f1_macro": f1_score(p.label_ids, pred, average="macro")}

pred_all["IndoBERT"] = pd.DataFrame(index=df.index)
os.makedirs(f"{FOLDER_HASIL}/cache", exist_ok=True)
gc.collect(); torch.cuda.empty_cache()

for aspek in ASPEK:
    f_ev = f"{FOLDER_HASIL}/cache/indobert_b{BATCH_INDOBERT}_{aspek}_eval.json"
    f_pr = f"{FOLDER_HASIL}/cache/indobert_b{BATCH_INDOBERT}_{aspek}_pred.csv"
    if PAKAI_CACHE and os.path.exists(f_ev) and os.path.exists(f_pr):
        print(f"[cache] IndoBERT {aspek} dimuat dari cache, dilewati")
        hasil_eval.append(json.load(open(f_ev)))
        pred_all["IndoBERT"][aspek] = pd.read_csv(f_pr)[aspek].values
        continue

    print(f"\n=== MENJALANKAN MODEL INDOBERT UNTUK ASPEK: {aspek.upper()} ===")
    set_seed(SEED)
    d_lat = data_latih.assign(y=data_latih[aspek].map(K2I))
    tr, va = train_test_split(d_lat, test_size=0.10, random_state=SEED, stratify=d_lat["y"])
    te = data_uji.assign(y=data_uji[aspek].map(K2I))
    al = df.assign(y=df[aspek].map(K2I))
    ds_tr, ds_va, ds_te, ds_all = (ke_dataset(x) for x in (tr, va, te, al))

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(KELAS), id2label=dict(enumerate(KELAS)), label2id=K2I)
    args = TrainingArguments(
        output_dir=f"./hasil_indobert_{aspek.lower()}", eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-5, per_device_train_batch_size=BATCH_INDOBERT, per_device_eval_batch_size=16,
        num_train_epochs=4, weight_decay=0.01, load_best_model_at_end=True,
        metric_for_best_model="f1_macro", save_total_limit=1, fp16=torch.cuda.is_available(),
        report_to="none", seed=SEED, logging_steps=50)
    trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                      data_collator=collator, compute_metrics=metrik)
    trainer.train()

    p = trainer.predict(ds_te)
    y_pred = p.predictions.argmax(1)
    acc = float(accuracy_score(p.label_ids, y_pred))
    f1m = float(f1_score(p.label_ids, y_pred, average="macro", labels=range(len(KELAS)), zero_division=0))
    print(classification_report(p.label_ids, y_pred, labels=range(len(KELAS)), target_names=KELAS, digits=3, zero_division=0))
    rep = classification_report(p.label_ids, y_pred, labels=range(len(KELAS)), target_names=KELAS, zero_division=0, output_dict=True)
    print(f"[HASIL] IndoBERT {aspek}: akurasi={acc*100:.2f}% | macro-F1={f1m:.3f}")

    rec = {"Model": "IndoBERT", "Aspek": aspek, "Akurasi": acc, "Macro-F1": f1m,
           **{f"F1 {k}": float(rep[k]["f1-score"]) for k in KELAS}}
    pred_aspek = [KELAS[i] for i in trainer.predict(ds_all).predictions.argmax(1)]
    hasil_eval.append(rec); pred_all["IndoBERT"][aspek] = pred_aspek
    json.dump(rec, open(f_ev, "w")); pd.DataFrame({aspek: pred_aspek}).to_csv(f_pr, index=False)   # simpan per aspek

    if SIMPAN_MODEL_INDOBERT:
        trainer.save_model(f"model_indobert_{aspek.lower()}")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

## 7. Perbandingan BiLSTM vs IndoBERT
Dibandingkan pada **data uji yang sama** (20%). Metrik utama adalah **macro-F1** (rata-rata F1 keempat kelas), karena label "-" mendominasi data.
Perhatikan bahwa jumlah contoh Negatif/Positif pada data uji kecil (puluhan, kadang hanya 3), sehingga F1 per kelas bisa berfluktuasi.

In [ ]:
eval_df = pd.DataFrame(hasil_eval)
tab = eval_df.pivot(index="Aspek", columns="Model", values=["Akurasi", "Macro-F1"]).reindex(ASPEK)
tab.loc["Rata-rata"] = tab.mean()
print((tab * 1).round(3).to_string())

pemenang = eval_df.groupby("Model")["Macro-F1"].mean().sort_values(ascending=False)
print("\nRata-rata macro-F1:", pemenang.round(3).to_dict())
print("Model terbaik berdasarkan rata-rata macro-F1:", pemenang.index[0])

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, met in zip(axs, ["Akurasi", "Macro-F1"]):
    eval_df.pivot(index="Aspek", columns="Model", values=met).reindex(ASPEK).plot(kind="bar", ax=ax, rot=0)
    ax.set_title(met); ax.set_ylim(0, 1.05); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(f"{FOLDER_HASIL}/perbandingan_model.png", dpi=150); plt.show()
eval_df.round(3)

## 7b. Confusion Matrix dan Uji McNemar
*Confusion matrix* menunjukkan kelas mana yang sering tertukar (misalnya Netral ditebak "-"). Uji McNemar memeriksa apakah selisih kinerja BiLSTM dan IndoBERT pada data uji yang sama **signifikan secara statistik** (bukan kebetulan). Statistiknya dihitung dari dua angka: *b* = komentar yang benar menurut BiLSTM tetapi salah menurut IndoBERT, dan *c* = sebaliknya. Dipakai versi *exact* (uji binomial), cocok untuk jumlah kasus kecil. Selisih dianggap signifikan bila p < 0,05.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scipy.stats import binomtest

idx_uji = data_uji.index
fig, axs = plt.subplots(2, 4, figsize=(17, 8))
baris_mcn = []
for r, m in enumerate(["BiLSTM", "IndoBERT"]):
    for c, a in enumerate(ASPEK):
        y_true = df.loc[idx_uji, a].values
        y_pred = pred_all[m].loc[idx_uji, a].values
        cm = confusion_matrix(y_true, y_pred, labels=KELAS)
        ConfusionMatrixDisplay(cm, display_labels=KELAS).plot(ax=axs[r, c], colorbar=False, cmap="Blues", values_format="d")
        axs[r, c].set_title(f"{m} - {a}")
        axs[r, c].set_xlabel("Prediksi"); axs[r, c].set_ylabel("Label asli" if c == 0 else "")
plt.tight_layout(); plt.savefig(f"{FOLDER_HASIL}/confusion_matrix.png", dpi=150); plt.show()

for a in ASPEK:
    y_true = df.loc[idx_uji, a].values
    benar_bl = pred_all["BiLSTM"].loc[idx_uji, a].values == y_true
    benar_ib = pred_all["IndoBERT"].loc[idx_uji, a].values == y_true
    b = int((benar_bl & ~benar_ib).sum())      # BiLSTM benar, IndoBERT salah
    c_ = int((~benar_bl & benar_ib).sum())     # BiLSTM salah, IndoBERT benar
    p = binomtest(min(b, c_), b + c_, 0.5).pvalue if (b + c_) > 0 else 1.0
    baris_mcn.append({"Aspek": a, "BiLSTM benar, IndoBERT salah (b)": b,
                      "BiLSTM salah, IndoBERT benar (c)": c_, "p-value": round(p, 4),
                      "Signifikan (p<0,05)": "Ya" if p < 0.05 else "Tidak"})
mcnemar_df = pd.DataFrame(baris_mcn)
mcnemar_df.to_csv(f"{FOLDER_HASIL}/uji_mcnemar.csv", index=False)
print("UJI McNEMAR (exact) BiLSTM vs IndoBERT pada data uji")
print(mcnemar_df.to_string(index=False))

## 8. Prediksi Sentimen Seluruh Komentar (masukan sistem rekomendasi)
Model terpilih (`MODEL_REKOMENDASI`) memprediksi aspek-sentimen seluruh komentar. Kolom `Split` menandai mana yang termasuk data uji.

**Catatan keterbatasan:** sekitar 80% komentar adalah data latih, sehingga prediksi pada bagian itu cenderung lebih akurat daripada pada data baru.
Evaluasi performa tetap hanya dilaporkan dari data uji (tahap 7).

In [ ]:
df_pred = df[["Video_ID", "Username", "Teks_Ulasan", "Teks_Bersih", "Split"]].copy()
for a in ASPEK:
    df_pred[a] = pred_all[MODEL_REKOMENDASI][a].values
nama_pred = f"{FOLDER_HASIL}/absa_prediksi_{MODEL_REKOMENDASI.lower()}.csv"
df_pred.to_csv(nama_pred, index=False)
print("Model untuk rekomendasi :", MODEL_REKOMENDASI)
print("Tersimpan               :", nama_pred)
print("\nDistribusi prediksi:"); print(pd.DataFrame({a: df_pred[a].value_counts() for a in ASPEK}).reindex(KELAS).fillna(0).astype(int))
df_pred.head()

## 9. Ekstraksi Merek dan Model Smartphone
Nama produk dibaca dari **`Teks_Ulasan`** (teks asli) memakai pola regex yang menangani penulisan informal (mis. `ip15pm`, `s25u`, `promax`, `16e`).
Aturan atribusi dibuat konservatif agar tidak ada produk "ngawur":

- Komentar dipakai untuk **ranking model** hanya jika tepat satu model disebut.
- Komentar dipakai untuk **ranking merek** jika tepat satu merek disebut.
- Komentar yang menyebut beberapa produk (perbandingan) atau tidak menyebut produk tidak dihitung.

Tabel `audit_atribusi.csv` menyimpan teks asli, hasil ekstraksi, dan status setiap komentar, supaya setiap angka di tabel rekomendasi bisa ditelusuri.

In [ ]:
import re, unicodedata

ASPEK = ["Baterai", "Kamera", "Performa", "Harga"]

# ----------------------------------------------------------------------------
# KONFIGURASI
# ----------------------------------------------------------------------------
MIN_OPINI_MODEL = 3      # model dengan opini < ini tidak ditampilkan
MIN_OPINI_MEREK = 10     # merek dengan opini < ini tidak ditampilkan
TOP_N = 5
K_SMOOTH = 5             # pseudo-opini netral untuk mengoreksi sampel kecil
GABUNG_VARIAN = False    # True: iPhone 16 Pro Max -> "iPhone 16" (naikkan N, kurangi presisi)

In [ ]:
# ----------------------------------------------------------------------------
# 1. EKSTRAKSI ENTITAS PRODUK
# ----------------------------------------------------------------------------
def _norm(t):
    t = unicodedata.normalize("NFKC", str(t)).lower()
    t = re.sub(r"https?://\S+|@[\w\-.]+", " ", t)
    return re.sub(r"(.)\1{3,}", r"\1\1", t)

def _var(v):
    if not v:
        return ""
    v = re.sub(r"\s+", "", v)
    return {"promax": "Pro Max", "pm": "Pro Max", "pro": "Pro", "pro+": "Pro+", "plus": "Plus",
            "+": "Plus", "mini": "Mini", "e": "e", "air": "Air", "ultra": "Ultra", "u": "Ultra",
            "fe": "FE", "edge": "Edge", "xl": "XL", "gt": "GT", "t": "T", "c": "C"}.get(v, v.title())

_NB = r"(?<![a-z0-9])"      # batas kiri
_NE = r"(?![a-z0-9])"       # batas kanan

# (grup_merek, regex, fungsi -> (seri, varian) | None)
POLA = [
    ("Apple", re.compile(_NB + r"(?:iphone|iphon|ip|ipon|aipon|ifon)\s?-?(\d{1,2})\s?"
                          r"(pro\s?max|promax|pm|pro|plus|mini|e|air)?" + _NE),
     lambda m: (f"iPhone {m[1]}", _var(m[2])) if 6 <= int(m[1]) <= 19 else None),
    ("Apple", re.compile(_NB + r"iphone\s?air" + _NE), lambda m: ("iPhone Air", "")),
    ("Samsung", re.compile(_NB + r"(?:samsung|galaxy|sam|sg)?\s?s\s?-?(2\d)\s?"
                                 r"(ultra|plus|\+|fe|edge|(?<=\d)u(?![a-z]))?" + _NE),
     lambda m: (f"Galaxy S{m[1]}", _var(m[2])) if 20 <= int(m[1]) <= 29 else None),
    ("Samsung", re.compile(_NB + r"(?:samsung|galaxy|sam)\s?a\s?-?(\d{2})" + _NE),
     lambda m: (f"Galaxy A{m[1]}", "")),
    ("Samsung", re.compile(_NB + r"(?:z\s?)?(fold|flip)\s?(\d)?" + _NE),
     lambda m: (f"Galaxy Z {m[1].title()}" + (f" {m[2]}" if m[2] else ""), "")),
    ("Xiaomi Group", re.compile(_NB + r"(?:xiaomi|xiomi|shaomi)\s?-?(1\d)(t)?\s?(pro|ultra)?" + _NE),
     lambda m: (f"Xiaomi {m[1]}{'T' if m[2] else ''}", _var(m[3]))),
    ("Xiaomi Group", re.compile(_NB + r"redmi\s?(note)?\s?-?(\d{1,2})\s?"
                                     r"(c|a|s|t|pro\s?max|promax|pro\s?\+|pro|plus|ultra)?" + _NE),
     lambda m: (("Redmi Note " if m[1] else "Redmi ") + m[2], _var(m[3]))),
    ("Xiaomi Group", re.compile(_NB + r"poco\s?([xfmc])\s?-?(\d{1,2})\s?(pro|gt|ultra)?" + _NE),
     lambda m: (f"Poco {m[1].upper()}{m[2]}", _var(m[3]))),
    ("Realme", re.compile(_NB + r"realme\s?(gt|c|narzo|p)?\s?-?(\d{1,3})\s?(pro\s?\+|pro|plus)?" + _NE),
     lambda m: ("Realme " + ((m[1].upper() + " ") if m[1] in ("gt", "narzo") else (m[1] or "").upper()) + m[2], _var(m[3]))),
    ("Vivo/iQOO", re.compile(_NB + r"vivo\s?([vxyt])\s?-?(\d{1,3})\s?(pro|e)?" + _NE),
     lambda m: (f"Vivo {m[1].upper()}{m[2]}", _var(m[3]))),
    ("Oppo", re.compile(_NB + r"oppo\s?(reno|find\s?[x×]|a)\s?-?(\d{1,2})\s?(pro|f)?" + _NE),
     lambda m: ((f"Oppo Find X{m[2]}" if "find" in m[1] else
                 f"Oppo A{m[2]}" if m[1] == "a" else f"Oppo Reno {m[2]}"), _var(m[3]))),
    ("OnePlus", re.compile(_NB + r"(?:oneplus|one\s?plus)\s?(\d{1,2})\s?(pro|r|t)?" + _NE),
     lambda m: (f"OnePlus {m[1]}", _var(m[2])) if int(m[1]) >= 9 else None),
    ("Google Pixel", re.compile(_NB + r"pixel\s?(\d{1,2})\s?(pro|xl|a)?" + _NE),
     lambda m: (f"Pixel {m[1]}", _var(m[2]))),
    ("Infinix/Tecno", re.compile(_NB + r"infinix\s?(hot|note|gt|zero|smart)\s?(\d{1,2})?\s?(pro|plus)?" + _NE),
     lambda m: (f"Infinix {m[1].title()}" + (f" {m[2]}" if m[2] else ""), _var(m[3]))),
]

# Kata kunci merek (tanpa angka) -> dipakai untuk ranking tingkat merek
KATA_MEREK = [
    ("Apple", r"iphone|\bios\b|apple|aipon|ipon|\bip\b"),
    ("Samsung", r"samsung|galaxy|\bsam\b|one\s?ui"),
    ("Xiaomi Group", r"xiaomi|redmi|poco|hyperos|mi fans"),
    ("Oppo", r"\boppo\b"), ("Vivo/iQOO", r"\bvivo\b|iqoo"), ("Realme", r"realme"),
    ("Infinix/Tecno", r"infinix|tecno|itel"), ("Honor", r"\bhonor\b"),
    ("OnePlus", r"oneplus|one plus"), ("Google Pixel", r"pixel"), ("Motorola", r"motorola|\bmoto\b"),
]

def ekstrak_entitas(teks):
    """-> (set_model, set_merek). Model dinormalisasi, merek = grup merek."""
    t = _norm(teks)
    model, merek = set(), set()
    for grup, rx, fn in POLA:
        for m in rx.finditer(t):
            r = fn(m)
            if r:
                seri, var = r
                if GABUNG_VARIAN or not var:
                    nama = seri
                else:
                    nama = seri + var if var in ("e", "C", "A", "S") else f"{seri} {var}"
                model.add(nama); merek.add(grup)
    for grup, rx in KATA_MEREK:
        if re.search(rx, t):
            merek.add(grup)
    return model, merek

In [ ]:
# ----------------------------------------------------------------------------
# 3. ATRIBUSI PRODUK
# ----------------------------------------------------------------------------
def atribusi(df):
    ent = df["Teks_Asli"].apply(ekstrak_entitas)
    df["Set_Model"] = ent.str[0]; df["Set_Merek"] = ent.str[1]
    n_model, n_merek = df["Set_Model"].str.len(), df["Set_Merek"].str.len()
    df["Model"] = [next(iter(m)) if (len(m) == 1 and len(b) == 1) else None
                   for m, b in zip(df["Set_Model"], df["Set_Merek"])]
    df["Merek"] = [next(iter(b)) if len(b) == 1 else None for b in df["Set_Merek"]]
    df["Status_Atribusi"] = "tidak ada produk"
    df.loc[n_merek > 1, "Status_Atribusi"] = "ambigu (multi produk)"
    df.loc[(n_merek == 1) & (n_model == 0), "Status_Atribusi"] = "merek saja"
    df.loc[df["Model"].notna(), "Status_Atribusi"] = "model teridentifikasi"
    return df

df_rek = df_pred.copy()
df_rek["Teks_Asli"] = df_rek["Teks_Ulasan"]
df_rek = atribusi(df_rek)

asp_ada = df_rek[(df_rek[ASPEK] != "-").any(axis=1)]
print(f"Komentar yang memuat aspek: {len(asp_ada)}")
print(asp_ada["Status_Atribusi"].value_counts().to_string())

## 10. Sistem Rekomendasi
Untuk setiap aspek dan setiap merek/model dihitung jumlah opini Positif, Netral, dan Negatif, lalu:

**Skor = (Positif - Negatif) / (N + 5)**

`N` adalah jumlah opini. Angka 5 menahan sampel kecil agar tidak menang kebetulan (mis. 3 dari 3 positif tidak otomatis juara). Tingkat keyakinan: **Kuat** (N >= 10), **Cukup** (5-9), **Indikatif** (3-4).
Merek ditampilkan jika N >= 10, model jika N >= 3; pada ringkasan akhir, model terbaik hanya ditampilkan bila N >= 5.

In [ ]:
# ----------------------------------------------------------------------------
# 4. AGREGASI & PERINGKAT
# ----------------------------------------------------------------------------
def keyakinan(n):
    return "Kuat" if n >= 10 else "Cukup" if n >= 5 else "Indikatif"

def rekap(df, aspek, kolom, min_n):
    d = df[(df[aspek] != "-") & df[kolom].notna()]
    if d.empty:
        return pd.DataFrame()
    t = d.groupby(kolom)[aspek].value_counts().unstack(fill_value=0)
    for c in ["Positif", "Netral", "Negatif"]:
        if c not in t: t[c] = 0
    t["N"] = t[["Positif", "Netral", "Negatif"]].sum(axis=1)
    t = t[t["N"] >= min_n].copy()
    if t.empty:
        return t
    t["% Positif"] = (100 * t["Positif"] / t["N"]).round(1)
    t["% Netral"] = (100 * t["Netral"] / t["N"]).round(1)
    t["% Negatif"] = (100 * t["Negatif"] / t["N"]).round(1)
    t["Skor"] = ((t["Positif"] - t["Negatif"]) / (t["N"] + K_SMOOTH)).round(3)
    t["Keyakinan"] = t["N"].apply(keyakinan)
    t = t.sort_values(["Skor", "N"], ascending=False).reset_index()
    t.insert(0, "Peringkat", range(1, len(t) + 1))
    t = t.rename(columns={kolom: kolom, "N": "Jumlah Opini"})
    return t[["Peringkat", kolom, "Jumlah Opini", "Positif", "Netral", "Negatif",
              "% Positif", "% Netral", "% Negatif", "Skor", "Keyakinan"]]

def bangun_semua(df):
    out = {}
    for a in ASPEK:
        out[("Merek", a)] = rekap(df, a, "Merek", MIN_OPINI_MEREK)
        out[("Model", a)] = rekap(df, a, "Model", MIN_OPINI_MODEL)
    return out

def ringkasan_final(out, min_n_ringkasan_model=5):
    """Merek terbaik = peringkat 1. Model terbaik hanya dipilih bila n >= 5 (selain itu: data belum cukup)."""
    baris = []
    for a in ASPEK:
        r = {"Aspek": a}
        for lvl in ["Merek", "Model"]:
            t = out[(lvl, a)]
            if lvl == "Model" and not t.empty:
                t = t[t["Jumlah Opini"] >= min_n_ringkasan_model]
            if t.empty:
                r[f"Terbaik ({lvl})"] = "Data belum cukup"; r[f"Keyakinan {lvl}"] = "-"
            else:
                top = t.iloc[0]
                r[f"Terbaik ({lvl})"] = f"{top[lvl]} (skor {top['Skor']:+.2f}, n={top['Jumlah Opini']})"
                r[f"Keyakinan {lvl}"] = top["Keyakinan"]
        baris.append(r)
    return pd.DataFrame(baris)

def laporan_cakupan(df):
    asp = df[(df[ASPEK] != "-").any(axis=1)]
    t = asp["Status_Atribusi"].value_counts().rename_axis("Status Atribusi").reset_index(name="Jumlah Komentar")
    t["Persen (%)"] = (100 * t["Jumlah Komentar"] / len(asp)).round(1)
    return t, len(asp)

In [ ]:
out = bangun_semua(df_rek)
ringkas = ringkasan_final(out)
cakupan, n_asp = laporan_cakupan(df_rek)

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("RINGKASAN REKOMENDASI FINAL"); print(ringkas.to_string(index=False))
for lvl in ["Merek", "Model"]:
    for a in ASPEK:
        t = out[(lvl, a)]
        print(f"\n--- TOP {lvl.upper()} | ASPEK {a.upper()} ---")
        print(t.head(TOP_N).to_string(index=False) if not t.empty else "(data belum mencukupi)")

### Uji sensitivitas (opsional, sangat disarankan untuk dilampirkan di Bab 4)
Apakah juara tiap aspek berubah bila sumber sentimen diganti? Tabel ini membandingkan merek terbaik berdasarkan **label asli**, **BiLSTM**, dan **IndoBERT**.
Jika hasilnya berbeda, itu menandakan peringkat peka terhadap kesalahan model dan sebaiknya diinterpretasikan sebagai kecenderungan, bukan kebenaran mutlak.

In [ ]:
rows = []
sumber = {"Label asli": df[ASPEK], "BiLSTM": pred_all["BiLSTM"][ASPEK], "IndoBERT": pred_all["IndoBERT"][ASPEK]}
for nama, lab in sumber.items():
    d = df_rek.copy()
    for a in ASPEK:
        d[a] = lab[a].values
    r = ringkasan_final(bangun_semua(d)).set_index("Aspek")["Terbaik (Merek)"]
    rows.append(r.rename(nama))
sens = pd.concat(rows, axis=1)
print(sens.to_string())

## 10b. Rekomendasi Berbobot Sesuai Prioritas Pengguna
Bagian 10 menjawab *"siapa terbaik di tiap aspek"*. Bagian ini menjawab *"HP mana yang paling cocok untuk saya"* dengan menggabungkan keempat aspek sesuai bobot prioritas pengguna:

**Skor_akhir = Σ (bobot_aspek × Skor_aspek)**, dengan total bobot = 1 (bobot dinormalisasi otomatis).

- Skor_aspek diambil dari rumus bagian 10: (Positif − Negatif) / (N + 5).
- Aspek yang tidak punya opini untuk suatu produk diberi skor 0 (netral).
- Produk hanya ditampilkan bila total opininya mencukupi (merek ≥ 30, model ≥ 5 pada contoh profil), dan diberi label keyakinan berdasarkan total opini. Batas merek dinaikkan dari 10 menjadi 30 karena merek dengan sedikit opini (mis. 11 opini yang terbagi ke 4 aspek) sempat menempati peringkat 1 hanya karena kebetulan.

In [ ]:
MIN_OPINI_BERBOBOT_MEREK = 30   # mode berbobot: merek perlu >= 30 opini total agar peringkat stabil

def tabel_skor(df, tingkat):
    # Skor tiap aspek untuk setiap merek/model (semua produk dengan >= 1 opini)
    potong = []
    for a in ASPEK:
        t = rekap(df, a, tingkat, 1)
        if t.empty:
            continue
        t = t.set_index(tingkat)[["Positif", "Netral", "Negatif", "Jumlah Opini", "Skor"]]
        t.columns = [f"{k} {a}" for k in ["Pos", "Net", "Neg", "N", "Skor"]]
        potong.append(t)
    tab = pd.concat(potong, axis=1).fillna(0)
    for a in ASPEK:
        for k in ["Pos", "Net", "Neg", "N", "Skor"]:
            if f"{k} {a}" not in tab:
                tab[f"{k} {a}"] = 0
        for k in ["Pos", "Net", "Neg", "N"]:
            tab[f"{k} {a}"] = tab[f"{k} {a}"].astype(int)
        tab[f"Skor {a}"] = tab[f"Skor {a}"].astype(float)
    tab.index.name = "Nama"
    tab["Total Opini"] = tab[[f"N {a}" for a in ASPEK]].sum(axis=1)
    if tingkat == "Model":   # merek pemilik setiap model
        tab["Merek"] = df.dropna(subset=["Model"]).groupby("Model")["Merek"].agg(lambda s: s.mode().iat[0])
    else:
        tab["Merek"] = tab.index
    return tab

SKOR = {"Merek": tabel_skor(df_rek, "Merek"), "Model": tabel_skor(df_rek, "Model")}

def rekomendasi_berbobot(bobot, tingkat="Merek", top_n=5, filter_merek=None, min_opini=None):
    # bobot: dict aspek -> angka (mis. persen). Dinormalisasi agar totalnya 1.
    w = {a: float(bobot.get(a, 0)) for a in ASPEK}
    total = sum(w.values())
    assert total > 0, "Minimal satu aspek harus diberi bobot > 0"
    w = {a: v / total for a, v in w.items()}
    min_n = min_opini if min_opini is not None else (MIN_OPINI_BERBOBOT_MEREK if tingkat == "Merek" else MIN_OPINI_MODEL)
    t = SKOR[tingkat].copy()
    t = t[t["Total Opini"] >= min_n]
    if filter_merek:
        t = t[t["Merek"].isin(filter_merek)]
    t["Skor Akhir"] = sum(w[a] * t[f"Skor {a}"] for a in ASPEK)
    t["Keyakinan"] = t["Total Opini"].apply(keyakinan)
    t = t.sort_values(["Skor Akhir", "Total Opini"], ascending=False).head(top_n).reset_index()
    if tingkat == "Merek":
        t = t.drop(columns="Merek")
    t = t.rename(columns={"Nama": tingkat})
    t.insert(0, "Peringkat", range(1, len(t) + 1))
    kolom = ["Peringkat", tingkat] + (["Merek"] if tingkat == "Model" else []) + \
            ["Skor Akhir"] + [f"Skor {a}" for a in ASPEK] + ["Total Opini", "Keyakinan"]
    return t[kolom].round(3)

# Contoh profil pengguna (bobot dalam persen)
PROFIL = {
    "Seimbang":       {"Baterai": 25, "Kamera": 25, "Performa": 25, "Harga": 25},
    "Pecinta kamera": {"Baterai": 20, "Kamera": 50, "Performa": 20, "Harga": 10},
    "Gamer":          {"Baterai": 30, "Kamera": 10, "Performa": 50, "Harga": 10},
    "Hemat":          {"Baterai": 25, "Kamera": 10, "Performa": 15, "Harga": 50},
}
hasil_profil = []
for nama, bobot in PROFIL.items():
    hasil = rekomendasi_berbobot(bobot, "Merek", top_n=5)
    print(f"\n=== PROFIL: {nama.upper()}  bobot={bobot} ===")
    print(hasil.to_string(index=False))
    hasil_profil.append(hasil.assign(Profil=nama))
    hasil_m = rekomendasi_berbobot(bobot, "Model", top_n=3, min_opini=5)
    print(f"--- Top 3 model (total opini >= 5) ---")
    print(hasil_m.to_string(index=False) if not hasil_m.empty else "(data belum cukup)")
pd.concat(hasil_profil).to_csv(f"{FOLDER_HASIL}/contoh_rekomendasi_berbobot.csv", index=False)
SKOR["Merek"].to_csv(f"{FOLDER_HASIL}/tabel_skor_merek.csv")
SKOR["Model"].to_csv(f"{FOLDER_HASIL}/tabel_skor_model.csv")

### 10c. Demo interaktif (opsional)
Geser slider bobot tiap aspek, pilih tingkat Merek atau Model, lalu lihat Top-N berubah secara langsung. Sel ini memakai `ipywidgets` yang sudah tersedia di Colab.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

sl = {a: widgets.IntSlider(value=25, min=0, max=100, step=5, description=a, continuous_update=False) for a in ASPEK}
pilih_tingkat = widgets.ToggleButtons(options=["Merek", "Model"], description="Tingkat")
keluaran = widgets.Output()

def perbarui(_=None):
    with keluaran:
        clear_output()
        bobot = {a: s.value for a, s in sl.items()}
        if sum(bobot.values()) == 0:
            print("Beri bobot > 0 pada minimal satu aspek."); return
        tingkat = pilih_tingkat.value
        display(rekomendasi_berbobot(bobot, tingkat, top_n=5, min_opini=None if tingkat == "Merek" else 5))

for w_ in list(sl.values()) + [pilih_tingkat]:
    w_.observe(perbarui, names="value")
display(widgets.VBox([widgets.HBox([widgets.VBox(list(sl.values())), pilih_tingkat]), keluaran]))
perbarui()

### 10d. Ekspor data untuk antarmuka (web/mobile)
Menyimpan `data_ui.json` berisi skor setiap merek dan model per aspek beserta beberapa contoh komentar (tanpa nama pengguna). File ini dipakai oleh prototipe antarmuka web sehingga angka yang tampil di aplikasi sama dengan hasil notebook.

In [ ]:
def contoh_komentar(df, tingkat, nama, aspek, n=2, maks=180):
    d = df[(df[tingkat] == nama) & (df[aspek] != "-")]
    d = pd.concat([d[d[aspek] == lab].head(1) for lab in ["Positif", "Negatif", "Netral"]]).head(n)
    return [{"label": r[aspek], "teks": (str(r["Teks_Asli"])[:maks] + ("..." if len(str(r["Teks_Asli"])) > maks else "")),
             "video": r["Video_ID"]} for _, r in d.iterrows()]

data_ui = {"aspek": ASPEK, "k_smooth": K_SMOOTH, "min_opini": {"Merek": MIN_OPINI_MEREK, "Model": MIN_OPINI_MODEL}, "min_opini_berbobot": {"Merek": MIN_OPINI_BERBOBOT_MEREK, "Model": 5},
           "model_sentimen": MODEL_REKOMENDASI, "jumlah_komentar": int(len(df_rek)),
           "evaluasi": eval_df[["Model", "Aspek", "Akurasi", "Macro-F1"]].round(4).to_dict("records"),
           "produk": []}
for tingkat in ["Merek", "Model"]:
    for nama, r in SKOR[tingkat].iterrows():
        if r["Total Opini"] < 3:
            continue
        data_ui["produk"].append({
            "nama": nama, "tingkat": tingkat, "merek": r["Merek"], "total_opini": int(r["Total Opini"]),
            "aspek": {a: {"pos": int(r[f"Pos {a}"]), "net": int(r[f"Net {a}"]), "neg": int(r[f"Neg {a}"]),
                          "n": int(r[f"N {a}"]), "skor": round(float(r[f"Skor {a}"]), 3),
                          "contoh": contoh_komentar(df_rek, tingkat, nama, a)} for a in ASPEK}})
with open(f"{FOLDER_HASIL}/data_ui.json", "w", encoding="utf-8") as f:
    json.dump(data_ui, f, ensure_ascii=False, indent=1)
print(f"data_ui.json tersimpan: {len(data_ui['produk'])} produk "
      f"({sum(p['tingkat']=='Merek' for p in data_ui['produk'])} merek, {sum(p['tingkat']=='Model' for p in data_ui['produk'])} model)")

## 11. Ekspor Hasil
Semua keluaran disimpan di folder `hasil/` dan dikemas menjadi satu file ZIP untuk diunduh:
`rekomendasi_smartphone.xlsx` (tabel siap Bab 4, termasuk perbandingan model dan sheet validasi manual), `audit_atribusi.csv`, `absa_prediksi_*.csv`, grafik, serta data latih/uji.

In [ ]:
# ----------------------------------------------------------------------------
# 5. EKSPOR
# ----------------------------------------------------------------------------
def tulis_excel(path, out, ringkas, cakupan, df, n_sampel=100, eval_df=None):
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    sampel = (df[df["Status_Atribusi"] != "tidak ada produk"]
              .sample(min(n_sampel, (df["Status_Atribusi"] != "tidak ada produk").sum()), random_state=42)
              [["Video_ID", "Teks_Asli", "Model", "Merek", "Status_Atribusi"]].copy())
    sampel["Benar? (isi Y/N)"] = ""
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        ringkas.to_excel(xw, sheet_name="Ringkasan Final", index=False)
        cakupan.to_excel(xw, sheet_name="Cakupan Atribusi", index=False)
        if eval_df is not None:
            eval_df.to_excel(xw, sheet_name="Perbandingan Model", index=False)
        for lvl in ["Merek", "Model"]:
            for a in ASPEK:
                t = out[(lvl, a)]
                (t.head(TOP_N) if not t.empty else pd.DataFrame({"Info": ["Data tidak mencukupi"]})
                 ).to_excel(xw, sheet_name=f"Top {lvl} - {a}"[:31], index=False)
        sampel.to_excel(xw, sheet_name="Sampel Validasi Entitas", index=False)
        for ws in xw.book.worksheets:
            for c in ws[1]:
                c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="1F4E78")
                c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            for col in ws.columns:
                w = max(len(str(c.value)) if c.value is not None else 0 for c in col[:60])
                ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(10, w + 2), 70)

def cetak(out, ringkas, cakupan, n_asp):
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
    print("=" * 100); print("CAKUPAN ATRIBUSI PRODUK (dari %d komentar yang memuat aspek)" % n_asp); print("=" * 100)
    print(cakupan.to_string(index=False))
    for lvl in ["Merek", "Model"]:
        for a in ASPEK:
            t = out[(lvl, a)]
            print("\n" + "-" * 100 + f"\nTOP REKOMENDASI {lvl.upper()} - ASPEK {a.upper()}\n" + "-" * 100)
            print(t.head(TOP_N).to_string(index=False) if not t.empty
                  else f"(tidak ada {lvl.lower()} dengan opini >= {MIN_OPINI_MEREK if lvl=='Merek' else MIN_OPINI_MODEL})")
    print("\n" + "=" * 100 + "\nRINGKASAN REKOMENDASI FINAL\n" + "=" * 100)
    print(ringkas.to_string(index=False))

tulis_excel(f"{FOLDER_HASIL}/rekomendasi_smartphone.xlsx", out, ringkas, cakupan, df_rek, eval_df=eval_df)
df_rek.drop(columns=["Set_Model", "Set_Merek"]).to_csv(f"{FOLDER_HASIL}/audit_atribusi.csv", index=False)
eval_df.to_csv(f"{FOLDER_HASIL}/evaluasi_model.csv", index=False)
for (lvl, asp_), t in out.items():
    t.to_csv(f"{FOLDER_HASIL}/top_{lvl.lower()}_{asp_.lower()}.csv", index=False)

shutil.make_archive("hasil_penelitian", "zip", FOLDER_HASIL)
print("Isi folder hasil:", sorted(os.listdir(FOLDER_HASIL)))
from google.colab import files
files.download("hasil_penelitian.zip")